In [1]:
from transformers import BertTokenizer, BertModel
import torch
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')
model

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
            (dropout): Dropout(p=

In [3]:
def get_sentence_embedding(text):
    inputs = tokenizer(text, return_tensors='pt')
    with torch.no_grad():
        outputs = model(**inputs)
    last_hidden_states = outputs.last_hidden_state
    sentence_embedding = torch.mean(last_hidden_states, dim=1).numpy()
    return sentence_embedding

In [4]:
texts = [
    "The quick brown fox jumps over the lazy dog.",
    "A fast brown fox leaps over a sleepy dog.",
    "This sentence is completely different from the others."
]

# Generate embeddings for texts
embeddings = [get_sentence_embedding(text) for text in texts]
embeds=np.array(embeddings)
embeds.shape

(3, 1, 768)

In [5]:
query_text = "The quick red fox jumps over the lazy dog."
query_embedding = get_sentence_embedding(query_text)
query_embedding[0][:20]

array([ 0.01581359, -0.06134035,  0.04332429,  0.03471805,  0.32235068,
        0.00439679, -0.12446141,  0.38215235,  0.10109728, -0.09531725,
       -0.0861407 , -0.16277878, -0.13426332, -0.05922271, -0.40886393,
       -0.19059594,  0.15335268, -0.12626094, -0.1464549 ,  0.00901168],
      dtype=float32)

In [6]:
similarities = cosine_similarity(query_embedding, np.vstack(embeddings))
similarities

array([[0.9852561, 0.9026968, 0.4810605]], dtype=float32)

In [7]:
print (f"Query text: {query_text}")
for i, text in enumerate(texts):
    print(f"Similarity with '{text}': {similarities[0][i]*100:.1f}%")

Query text: The quick red fox jumps over the lazy dog.
Similarity with 'The quick brown fox jumps over the lazy dog.': 98.5%
Similarity with 'A fast brown fox leaps over a sleepy dog.': 90.3%
Similarity with 'This sentence is completely different from the others.': 48.1%
